# Regularização: Ridge e Lasso

**Objetivo:** ver a penalidade em ação — traçar como os coeficientes encolhem com $\alpha$, contrastar Ridge (encolhe) e Lasso (zera) e escolher $\alpha$ por validação cruzada. Sempre com padronização.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

## 1. Dados e padronização

Conjunto diabetes de novo. Como a penalidade depende da escala dos coeficientes, **padronizamos** os preditores (média 0, desvio 1) — feito dentro de um `Pipeline` para não vazar informação do teste.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge, Lasso

dados = load_diabetes(as_frame=True)
X = dados.data.values
y = dados.target.values
nomes = list(dados.data.columns)
print("X:", X.shape, "| preditores:", nomes)

## 2. O caminho de regularização

Para uma faixa de $\alpha$ (em escala logarítmica), ajustamos Ridge e Lasso e guardamos os coeficientes. Dois laços à mostra: um por método, um por valor de $\alpha$.

In [ ]:
alphas = np.logspace(-2, 2, 40)

caminho_ridge = []
caminho_lasso = []
for a in alphas:
    ridge = make_pipeline(StandardScaler(), Ridge(alpha=a)).fit(X, y)
    lasso = make_pipeline(StandardScaler(), Lasso(alpha=a, max_iter=10000)).fit(X, y)
    caminho_ridge.append(ridge.named_steps["ridge"].coef_)
    caminho_lasso.append(lasso.named_steps["lasso"].coef_)
caminho_ridge = np.array(caminho_ridge)
caminho_lasso = np.array(caminho_lasso)
print("formato do caminho (n_alphas, n_preditores):", caminho_ridge.shape)

In [ ]:
from plotly.subplots import make_subplots

figura = make_subplots(rows=1, cols=2, subplot_titles=("Ridge (l2)", "Lasso (l1)"))
for j in range(X.shape[1]):
    figura.add_trace(go.Scatter(x=alphas, y=caminho_ridge[:, j], mode="lines",
                                name=nomes[j], showlegend=False), row=1, col=1)
    figura.add_trace(go.Scatter(x=alphas, y=caminho_lasso[:, j], mode="lines",
                                name=nomes[j]), row=1, col=2)
figura.update_xaxes(type="log", title_text="alpha (log)")
figura.update_yaxes(title_text="coeficiente", row=1, col=1)
figura.update_layout(title="Caminho de regularizacao: Ridge encolhe, Lasso zera",
                     height=400, margin=dict(l=10, r=10, t=60, b=10))
figura.show()

## 3. Lasso zera coeficientes

Contamos, para cada $\alpha$, quantos coeficientes o Lasso mantém diferentes de zero. Quanto maior $\alpha$, mais esparso o modelo.

In [ ]:
n_diferentes_de_zero = (np.abs(caminho_lasso) > 1e-6).sum(axis=1)
figura = go.Figure(go.Scatter(x=alphas, y=n_diferentes_de_zero, mode="lines+markers",
                              line=dict(color=VERDE)))
figura.update_xaxes(type="log", title_text="alpha (log)")
figura.update_layout(title="Lasso: numero de coeficientes != 0 vs alpha",
                     yaxis_title="coeficientes ativos", height=340,
                     margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## 4. Escolhendo α por validação cruzada

`RidgeCV` e `LassoCV` testam vários $\alpha$ por validação cruzada e ficam com o que minimiza o erro em dados não vistos — nada de escolher no olho.

In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV

ridge_cv = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas)).fit(X, y)
lasso_cv = make_pipeline(StandardScaler(), LassoCV(alphas=alphas, max_iter=10000)).fit(X, y)
print("melhor alpha Ridge:", round(ridge_cv.named_steps["ridgecv"].alpha_, 3))
print("melhor alpha Lasso:", round(lasso_cv.named_steps["lassocv"].alpha_, 3))
coef_lasso = lasso_cv.named_steps["lassocv"].coef_
print("preditores mantidos pelo Lasso:",
      [nomes[j] for j in range(len(coef_lasso)) if abs(coef_lasso[j]) > 1e-6])

## Exercício

No caminho do Lasso, qual preditor é o **último** a ser zerado quando $\alpha$ cresce? O que isso sugere sobre a importância dele?

In [ ]:
# @title Solução (clique para revelar)
a_grande = alphas[-8]
lasso_forte = make_pipeline(StandardScaler(), Lasso(alpha=a_grande, max_iter=10000)).fit(X, y)
coef = lasso_forte.named_steps["lasso"].coef_
sobreviventes = [(nomes[j], round(coef[j], 1)) for j in range(len(coef)) if abs(coef[j]) > 1e-6]
print("com alpha alto, sobrevivem:", sobreviventes)
# O ultimo a resistir e o preditor mais robustamente associado ao alvo:
# o Lasso o considera o mais informativo, o ultimo do qual abre mao.